In [1]:
import pandas as pd
import numpy as np
from pandas import IndexSlice

from config import PATHS

In [2]:
def print_stats(arr):
    print(f'Min : {arr.min()}')
    print(f'Mean: {arr.mean()}')
    print(f'Max : {arr.max()}')
    print(f'Std : {arr.std()}')

In [3]:
pi = pd.read_pickle(PATHS.patient_info_exact.pickle)
pi = pi[~pi.index.get_level_values('patient').str.contains('FAKE')]
# pi

In [ ]:
valid = pi[pi['valid']]
# valid

In [5]:
print_stats(valid['timespan'])

Min : 132 days 00:25:40
Mean: 350 days 11:50:59
Max : 537 days 20:22:38
Std : 114 days 18:26:31.480742408


In [6]:
print_stats(valid['valid_seizures'])

Min : 13
Mean: 54.5
Max : 195
Std : 48.608828229223775


In [ ]:
# For Viana et al. (2023) Seizure forecasting using minimally invasive, ultra-long-term subcutaneous electroencephalography: Individualized intrapatient models

In [19]:
timespans_days = [89, 69, 76, 84, 230, 46]
sum(timespans_days) / len(timespans_days)

99.0

In [131]:
# Architecture 3 (BiLSTM)
aucs = np.array([74, 59, 78, 64, 79, 74], dtype=float)

In [132]:
sensitivity = np.array([70, 67, 75, 73, 73, 80], dtype=float)

In [133]:
tiw = np.array([39.1, 48.1, 25.1, 42.2, 13.6, 36.3])

In [134]:
BiLSTM = pd.DataFrame({
    'Rel. TIFW': tiw,
    'EB Sensitivity': sensitivity,
    'ROC AUC': aucs
})
BiLSTM

,Rel. TIFW,EB Sensitivity,ROC AUC
0,39.1,70.0,74.0
1,48.1,67.0,59.0
2,25.1,75.0,78.0
3,42.2,73.0,64.0
4,13.6,73.0,79.0
5,36.3,80.0,74.0


In [135]:
# --------- For my results:
model_stats = pd.read_pickle(PATHS.per_model_comparison_table.pickle)
model_stats.drop(columns=['best_threshold', 'event_based_f1', 'precision', 'recall', 'p_hanley_mcneil'], inplace=True)
model_stats.rename(columns={'rel_tifw': 'Rel. TIFW', 'rel_szrs_pred': 'EB Sensitivity', 'roc_auc': 'ROC AUC'},
                   inplace=True)
model_stats *= 100  # convert to percent
# model_stats

/tmp/ipykernel_1864721/972477327.py:3: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  model_stats.drop(columns=['best_threshold', 'event_based_f1', 'precision', 'recall', 'p_hanley_mcneil'], inplace=True)


In [136]:
test = model_stats.loc[:, IndexSlice[:, 'test']].droplevel('split', axis='columns')
# test

In [137]:
cnn = test.loc[IndexSlice[:, 'CNN'], :].droplevel('model', axis='index')
ens = test.loc[IndexSlice[:, 'ensemble'], :].droplevel('model', axis='index')

In [139]:
print_stats(cnn['Rel. TIFW'])

Min : 16.504752951008044
Mean: 32.18654342827251
Max : 47.22660169127132


In [146]:
# Create table
out = pd.DataFrame(
    index=pd.MultiIndex.from_product([['min', 'mean', 'max'], ['BiLSTM', 'CNN', 'Ensemble']], names=['', 'Model']),
    columns=cnn.columns,
    dtype=float
)

assert BiLSTM.columns.to_list() == cnn.columns.to_list() == ens.columns.to_list() == out.columns.to_list()

for func in ['min', 'mean', 'max']:
    for model_name, model_df in [('BiLSTM', BiLSTM), ('CNN', cnn), ('Ensemble', ens)]:
        out.loc[(func, model_name), :] = getattr(model_df, func)()

out = out.round().astype(int)
out

metric         Rel. TIFW  EB Sensitivity  ROC AUC
     Model                                       
min  BiLSTM           14              67       59
     CNN              17              54       53
     Ensemble         19              40       38
mean BiLSTM           34              73       71
     CNN              32              71       68
     Ensemble         38              69       66
max  BiLSTM           48              80       79
     CNN              47              86       85
     Ensemble         60              96       85

In [153]:
r = out.copy()
r = r.astype(str)
r += r' \%'
r = r.rename(
    index={'min': 'Min', 'mean': 'Mean', 'max': 'Max'},
    columns={
        'Rel. TIFW': r'\textbf{Rel. TIFW}',
        'EB Sensitivity': r'\textbf{EB Sensitivity}',
        'ROC AUC': r'\textbf{ROC AUC}'
    })
r = r.rename_axis(index={'': None, 'Model': r'\textbf{Model}'},
                  columns={'': None, 'metric': r'\textbf{Metric:}'})
print(r.to_latex(column_format='llrrrr'))

\begin{tabular}{llrrrr}
\toprule
 & \textbf{Metric:} & \textbf{Rel. TIFW} & \textbf{EB Sensitivity} & \textbf{ROC AUC} \\
 & \textbf{Model} &  &  &  \\
\midrule
\multirow[t]{3}{*}{Min} & BiLSTM & 14 \% & 67 \% & 59 \% \\
 & CNN & 17 \% & 54 \% & 53 \% \\
 & Ensemble & 19 \% & 40 \% & 38 \% \\
\cline{1-5}
\multirow[t]{3}{*}{Mean} & BiLSTM & 34 \% & 73 \% & 71 \% \\
 & CNN & 32 \% & 71 \% & 68 \% \\
 & Ensemble & 38 \% & 69 \% & 66 \% \\
\cline{1-5}
\multirow[t]{3}{*}{Max} & BiLSTM & 48 \% & 80 \% & 79 \% \\
 & CNN & 47 \% & 86 \% & 85 \% \\
 & Ensemble & 60 \% & 96 \% & 85 \% \\
\cline{1-5}
\bottomrule
\end{tabular}

